# 🎙️ Training Whisper Small — Kaggle T4 x2 (version optimisée)
**Dataset multilingue : Arabe / Anglais / Français**
#
### Optimisations T4 x2 :
- DataParallel automatique sur les 2 GPUs via Trainer
- batch effectif = 64 (16 par GPU × 2 GPUs × grad_accum 2)
- dataloader_num_workers=4 (Linux)
- fp16=True
- gradient_checkpointing=True
- generation_max_length=128 (réduit pour accélérer les évals)

## Cellule 1 — Installation

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip uninstall -y transformers peft accelerate -q

!pip install -q \
transformers==4.40.2 \
peft==0.10.0 \
accelerate==0.29.3 \
datasets \
evaluate \
jiwer \
tensorboard



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 112.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.3 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.


## Cellule 2 — Config & Chemins

In [2]:
import json, glob, shutil
import numpy as np
import torch
from collections import Counter
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_from_disk
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
import evaluate
import transformers

# ── Vérification GPU ───────────────────────────────────────────────────────────
n_gpus  = torch.cuda.device_count()
HAS_GPU = n_gpus > 0

print(f"GPUs visibles    : {n_gpus}  (doit être 1)")
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i} : {name}  |  VRAM : {vram:.1f} Go")

print(f"\nPyTorch          : {torch.__version__}")
print(f"Transformers     : {transformers.__version__}  (doit être 4.40.0)")

if n_gpus != 1:
    raise RuntimeError(
        "❌ Encore 2 GPUs visibles — redémarre le kernel (Runtime > Restart)\n"
        "   puis relance depuis Cellule 1 sans sauter de cellule."
    )
print("✅ Configuration GPU correcte")

GPUs visibles    : 1  (doit être 1)
  GPU 0 : Tesla T4  |  VRAM : 15.6 Go

PyTorch          : 2.10.0+cu128
Transformers     : 4.40.2  (doit être 4.40.0)
✅ Configuration GPU correcte


## Cellule 3 — Chargement Dataset

In [5]:
DATASET_SLUG = "datasets/rihemochi/dataset-voc"  # ← ton slug exact
RESUME_SLUG  = None   # ← slug checkpoints session précédente (ou None)

FEATURES_DIR     = f"/kaggle/input/{DATASET_SLUG}/dataset_final"
OUTPUT_DIR       = "/kaggle/working/whisper-small-nexor"
MODEL_NAME       = "openai/whisper-small"
EVAL_MAX_SAMPLES = 400   # sous-ensemble pour évals rapides pendant training

# Batch optimisé T4 15 Go (1 GPU)
BATCH_TRAIN = 16
GRAD_ACCUM  = 4      # batch effectif = 16 × 4 = 64
BATCH_EVAL  = 8
EFFECTIVE   = BATCH_TRAIN * GRAD_ACCUM   # 64

assert os.path.isdir(FEATURES_DIR), (
    f"❌ Dataset introuvable : {FEATURES_DIR}\n"
    f"   → Vérifie DATASET_SLUG = '{DATASET_SLUG}'"
)

print(f"✅ Dataset         : {FEATURES_DIR}")
print(f"   Batch effectif  : {EFFECTIVE}  ({BATCH_TRAIN} × {GRAD_ACCUM})")
print(f"   Eval samples    : {EVAL_MAX_SAMPLES}")
print(f"   Output dir      : {OUTPUT_DIR}")


✅ Dataset         : /kaggle/input/datasets/rihemochi/dataset-voc/dataset_final
   Batch effectif  : 64  (16 × 4)
   Eval samples    : 400
   Output dir      : /kaggle/working/whisper-small-nexor


## Cellule 4 — Processor, DataCollator & WER

In [6]:
if RESUME_SLUG:
    resume_src = f"/kaggle/input/{RESUME_SLUG}/whisper-small-nexor"
    if os.path.isdir(resume_src):
        print(f"♻️  Copie checkpoints depuis : {resume_src}")
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        shutil.copytree(resume_src, OUTPUT_DIR, dirs_exist_ok=True)
        print("✅ Checkpoints copiés")
    else:
        print(f"⚠️  Résumption introuvable — démarrage zéro")

print("Chargement dataset...")
dataset = load_from_disk(FEATURES_DIR)

# Vérifications
assert "train"      in dataset, "❌ Split 'train' manquant"
assert "validation" in dataset, "❌ Split 'validation' manquant"
assert "test"       in dataset, "❌ Split 'test' manquant"
for col in ["input_features", "labels", "lang", "text"]:
    assert col in dataset["train"].column_names, f"❌ Colonne '{col}' manquante"

sample_shape = tuple(np.array(dataset["train"][0]["input_features"]).shape)
assert sample_shape == (80, 3000), f"❌ Shape inattendue : {sample_shape}"

print(f"✅ Dataset chargé :")
print(f"   train      : {len(dataset['train']):,}")
print(f"   validation : {len(dataset['validation']):,}")
print(f"   test       : {len(dataset['test']):,}")
print(f"   shape      : {sample_shape} ✅")

print(f"\n   Distribution langues :")
for split in ["train", "validation", "test"]:
    counts = Counter(dataset[split]["lang"])
    parts  = "  ".join(f"{l}={c:,}" for l, c in sorted(counts.items()))
    print(f"   {split:<12} : {parts}")

# Sous-ensemble eval rapide
val_ds = dataset["validation"]

langs = val_ds["lang"]

fr_idx = [i for i, l in enumerate(langs) if l == "fr"][:133]
en_idx = [i for i, l in enumerate(langs) if l == "en"][:133]
ar_idx = [i for i, l in enumerate(langs) if l == "ar"][:134]

eval_subset = val_ds.select(fr_idx + en_idx + ar_idx)

print(f"Eval subset : {len(eval_subset)} exemples")
print(f"\n   eval subset : {len(eval_subset)} exemples (évals rapides)")


Chargement dataset...


Loading dataset from disk:   0%|          | 0/126 [00:00<?, ?it/s]

✅ Dataset chargé :
   train      : 32,617
   validation : 4,019
   test       : 4,054
   shape      : (80, 3000) ✅

   Distribution langues :
   train        : ar=10,986  en=10,950  fr=10,681
   validation   : ar=1,304  en=1,363  fr=1,352
   test         : ar=1,421  en=1,280  fr=1,353
Eval subset : 400 exemples

   eval subset : 400 exemples (évals rapides)


## Cellule 5 — Modèle & Trainer

In [7]:
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
tokenizer = processor.tokenizer

# Vérification tokens langue — whisper-small utilise codes courts
print("Vérification tokens langue :")
lang_token_ids = {}
for lang_code in ["ar", "en", "fr"]:
    token_str = f"<|{lang_code}|>"
    token_id  = tokenizer.convert_tokens_to_ids(token_str)
    assert token_id != tokenizer.unk_token_id, f"❌ {token_str} introuvable !"
    lang_token_ids[lang_code] = token_id
    print(f"  ✅ {token_str} → id {token_id}")
assert len(set(lang_token_ids.values())) == 3, "❌ Collision token IDs !"

# Inspection labels — vérifie que les tokens langue sont présents
print("\n🔍 Inspection labels (3 premiers train) :")
all_ok = True
for i in range(3):
    ex      = dataset["train"][i]
    decoded = tokenizer.decode(ex["labels"], skip_special_tokens=False)
    lang    = ex["lang"]
    ok      = f"<|{lang}|>" in decoded
    print(f"  {'✅' if ok else '❌'} [{lang}] {decoded[:90]}")
    if not ok:
        all_ok = False

if not all_ok:
    raise ValueError(
        "❌ Labels sans token langue — refais le preprocessing !\n"
        "   Les labels doivent contenir <|ar|>, <|en|> ou <|fr|>"
    )

# DataCollator
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor             : Any
    decoder_start_token_id: int

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:
        input_feats  = [{"input_features": f["input_features"]} for f in features]
        batch        = self.processor.feature_extractor.pad(input_feats, return_tensors="pt")
        label_feats  = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_feats, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=tokenizer.bos_token_id,
)

sample = [dataset["train"][i] for i in range(4)]
batch  = data_collator(sample)
assert batch["input_features"].shape == (4, 80, 3000)
print(f"\n✅ DataCollator OK  |  batch : {tuple(batch['input_features'].shape)}")

# WER avec débogage
wer_metric    = evaluate.load("wer")
_eval_counter = [0]

def compute_metrics(pred):
    _eval_counter[0] += 1
    n = _eval_counter[0]

    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

    pred_str  = [p.strip() for p in tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)]
    label_str = [l.strip() for l in tokenizer.batch_decode(label_ids, skip_special_tokens=True)]

    # Affiche 3 paires aux 3 premiers appels pour débogage
    if n <= 3:
        print(f"\n📊 [Éval #{n}] Exemples décodés :")
        for i in range(min(3, len(pred_str))):
            print(f"  REF  : {label_str[i][:80]}")
            print(f"  PRED : {pred_str[i][:80]}")

    pairs = [(p, r) for p, r in zip(pred_str, label_str) if r.strip()]
    if not pairs:
        print(f"⚠️  [Éval #{n}] Références vides — WER=1.0")
        return {"wer": 1.0}

    preds_f, refs_f = zip(*pairs)
    wer = wer_metric.compute(predictions=list(preds_f), references=list(refs_f))

    if n <= 3:
        print(f"  → WER #{n} = {wer:.4f}  ({len(pairs)} exemples)")

    return {"wer": round(wer, 4)}

print("✅ WER metric OK")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Vérification tokens langue :
  ✅ <|ar|> → id 50272
  ✅ <|en|> → id 50259
  ✅ <|fr|> → id 50265

🔍 Inspection labels (3 premiers train) :
  ✅ [fr] <|startoftranscript|><|fr|><|transcribe|><|notimestamps|>Apportez la salade niçoise pour l
  ✅ [fr] <|startoftranscript|><|fr|><|transcribe|><|notimestamps|>Peut-on avoir la blanquette de ve
  ✅ [fr] <|startoftranscript|><|fr|><|transcribe|><|notimestamps|>Je souhaiterais commander la pann

✅ DataCollator OK  |  batch : (4, 80, 3000)


✅ WER metric OK


## Cellule 6 — TensorBoard

In [8]:
print(f"Chargement {MODEL_NAME}...")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.use_cache                        = False
model.config.forced_decoder_ids               = None
model.generation_config.forced_decoder_ids    = None
model.generation_config.suppress_tokens       = []
model.generation_config.begin_suppress_tokens = []
model.generation_config.language = None
model.generation_config.task     = "transcribe"
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ Modèle : {n_params:.0f}M paramètres")
print(f"   forced_decoder_ids : {model.generation_config.forced_decoder_ids}")
print(f"   suppress_tokens    : {model.generation_config.suppress_tokens}")

# Forward pass GPU
device = "cuda" if HAS_GPU else "cpu"
print(f"\nForward pass {device.upper()} (4 exemples)...")
model.eval()
model.to(device)
with torch.no_grad():
    out = model(
        input_features=batch["input_features"].to(device),
        labels=batch["labels"].to(device)
    )
assert out.loss is not None and out.loss.item() > 0
print(f"✅ Forward pass OK  |  loss={out.loss.item():.4f}")
model.cpu()

# TrainingArguments
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir                   = OUTPUT_DIR,
    per_device_train_batch_size  = BATCH_TRAIN,
    per_device_eval_batch_size   = BATCH_EVAL,
    gradient_accumulation_steps  = GRAD_ACCUM,
    learning_rate                = 5e-6,
    warmup_steps                 = 200,
    max_steps                    = 4000,
    lr_scheduler_type            = "linear",
    fp16                         = True,
    gradient_checkpointing       = True,
    gradient_checkpointing_kwargs= {"use_reentrant": False},
    dataloader_num_workers       = 4,
    dataloader_pin_memory        = True,
    evaluation_strategy          = "steps",
    eval_steps                   = 500,
    save_strategy                = "steps",
    save_steps                   = 500,
    save_total_limit             = 2,
    load_best_model_at_end       = True,
    metric_for_best_model        = "wer",
    greater_is_better            = False,
    predict_with_generate        = True,
    generation_max_length        = 128,
    logging_steps                = 25,
    report_to                    = ["tensorboard"],
    remove_unused_columns        = False,
)

print(f"\n✅ TrainingArguments OK")
print(f"   batch effectif : {EFFECTIVE}")
print(f"   lr             : {training_args.learning_rate}")
print(f"   max_steps      : {training_args.max_steps}")
print(f"   eval_steps     : {training_args.eval_steps}")
print(f"   eval sur       : {len(eval_subset)} exemples")
print(f"   gen_max_length : {training_args.generation_max_length}")

# ══════════════════════════════════════════════════════════════════
# FIX DÉFINITIF — "Multiple languages detected"
# ══════════════════════════════════════════════════════════════════
# CAUSE RACINE : Whisper 4.40 appelle _retrieve_init_tokens() AVANT
# de lire forced_decoder_ids. Quand language=None, il détecte la
# langue depuis l'audio → plusieurs langues dans le batch → crash.
# forced_decoder_ids défini après est ignoré car trop tard.
#
# SOLUTION : ne pas appeler super().prediction_step() pour la
# génération. On appelle model.generate() DIRECTEMENT avec
# `language=` explicite extrait des labels → _retrieve_init_tokens
# est court-circuité, Whisper utilise la langue qu'on lui donne.
# ══════════════════════════════════════════════════════════════════

# Mapping token_id → nom langue (pour passer à generate(language=...))
TOKEN_ID_TO_LANG = {
    tokenizer.convert_tokens_to_ids(f"<|{name}|>"): name
    for name in ["french", "arabic", "english"]
}

class WhisperTrainer(Seq2SeqTrainer):
    """
    Surcharge prediction_step pour appeler model.generate() directement
    avec language= extrait des labels de chaque sous-groupe monolingue.

    Fonctionnement :
      1. Lit les labels → détecte le token langue de chaque exemple
      2. Groupe les indices par langue
      3. Pour chaque groupe → appelle model.generate(language='french')
         (pas super().prediction_step → pas de _retrieve_init_tokens)
      4. Calcule la loss sur tout le batch via un forward pass normal
      5. Réassemble les tokens générés dans l'ordre original
    """

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        has_labels = "labels" in inputs

        # ── Loss (forward pass normal, pas de génération) ───────────
        with torch.no_grad():
            loss = None
            if has_labels:
                outputs = model(**inputs)
                loss    = outputs.loss.detach()

        if prediction_loss_only or not self.args.predict_with_generate:
            return (loss, None, None)

        # ── Génération multilingue ───────────────────────────────────
        labels         = inputs["labels"]          # (B, T)
        input_features = inputs["input_features"]  # (B, 80, 3000)
        B              = input_features.shape[0]

        # Étape 1 — détecte le token langue de chaque exemple
        # labels structure : [<|startoftranscript|>, <|french|>, <|transcribe|>, ...]
        lang_name_per_ex = []
        for row in labels:
            valid = [t.item() for t in row if t.item() != -100]
            lang_tok  = valid[1] if len(valid) > 1 else None
            lang_name = TOKEN_ID_TO_LANG.get(lang_tok, "english")  # fallback english
            lang_name_per_ex.append(lang_name)

        # Étape 2 — groupe par langue
        from collections import defaultdict
        groups = defaultdict(list)
        for idx, lname in enumerate(lang_name_per_ex):
            groups[lname].append(idx)

        # Étape 3 — génère séparément par groupe avec language= explicite
        generated_per_idx = {}
        gen_max = self.args.generation_max_length or 225

        for lang_name, indices in groups.items():
            sub_features = input_features[indices]  # (k, 80, 3000)
            with torch.no_grad():
                gen_out = model.generate(
                    sub_features,
                    language     = lang_name,   # ← court-circuite _retrieve_init_tokens
                    task         = "transcribe",
                    max_new_tokens = gen_max,
                    forced_decoder_ids    = None,
                    suppress_tokens       = [],
                    begin_suppress_tokens = [],
                )
            for out_pos, orig_idx in enumerate(indices):
                generated_per_idx[orig_idx] = gen_out[out_pos]

        # Étape 4 — réassemble dans l'ordre original + pad à longueur max
        gen_list = [generated_per_idx[i] for i in range(B)]
        max_len  = max(t.shape[0] for t in gen_list)
        pad_id   = tokenizer.pad_token_id

        generated_tokens = torch.stack([
            torch.nn.functional.pad(
                t, (0, max_len - t.shape[0]), value=pad_id
            )
            for t in gen_list
        ])  # (B, max_len)

        # Étape 5 — prépare les labels pour compute_metrics
        label_ids = labels.clone()
        label_ids[label_ids == -100] = pad_id
        max_label_len = label_ids.shape[1]
        if max_len < max_label_len:
            # Pad generated à la longueur des labels
            generated_tokens = torch.nn.functional.pad(
                generated_tokens, (0, max_label_len - max_len), value=pad_id
            )

        return (loss, generated_tokens, label_ids)

# Résumption automatique depuis dernier checkpoint
checkpoints = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")))
resume_from = checkpoints[-1] if checkpoints else None
print(f"\n   Reprise depuis : {resume_from or 'zéro'}")

trainer = WhisperTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = dataset["train"],
    eval_dataset     = eval_subset,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=4)],
)

print(f"✅ Trainer prêt")
print(f"   train  : {len(trainer.train_dataset):,} ex")
print(f"   eval   : {len(trainer.eval_dataset):,} ex")


Chargement openai/whisper-small...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

✅ Modèle : 242M paramètres
   forced_decoder_ids : None
   suppress_tokens    : []

Forward pass CUDA (4 exemples)...
✅ Forward pass OK  |  loss=3.4435

✅ TrainingArguments OK
   batch effectif : 64
   lr             : 5e-06
   max_steps      : 4000
   eval_steps     : 500
   eval sur       : 400 exemples
   gen_max_length : 128

   Reprise depuis : zéro


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


✅ Trainer prêt
   train  : 32,617 ex
   eval   : 400 ex


In [ ]:
load_ext tensorboard
tensorboard --logdir /kaggle/working/whisper-small-nexor

## Cellule 7 — 🚀 Entraînement
⏱️ Durée estimée T4 x2 : ~2h30 pour 4000 steps

In [9]:
print("🚀 Démarrage entraînement...")
print(f"   GPU             : {torch.cuda.get_device_name(0) if HAS_GPU else 'CPU'}")
print(f"   Batch effectif  : {EFFECTIVE}")
print(f"   Steps           : {training_args.max_steps}")
print(f"   Eval tous les   : {training_args.eval_steps} steps")
print(f"   Eval sur        : {len(eval_subset)} exemples")
print(f"   lr              : {training_args.learning_rate}")
print(f"   EarlyStopping   : patience=4")
print(f"   Reprise         : {resume_from or 'zéro'}")
print("─" * 55)
print("Vitesse attendue : ~0.3 it/s")
print("Si toujours < 0.1 it/s à step 30 → envoie une capture")
print("─" * 55)

trainer.train(resume_from_checkpoint=resume_from)
print("\n✅ Entraînement terminé !")


🚀 Démarrage entraînement...
   GPU             : Tesla T4
   Batch effectif  : 64
   Steps           : 4000
   Eval tous les   : 500 steps
   Eval sur        : 400 exemples
   lr              : 5e-06
   EarlyStopping   : patience=4
   Reprise         : zéro
───────────────────────────────────────────────────────
Vitesse attendue : ~0.3 it/s
Si toujours < 0.1 it/s à step 30 → envoie une capture
───────────────────────────────────────────────────────


Step,Training Loss,Validation Loss,Wer
500,0.063600,0.063688,0.471200
1000,0.043800,0.047734,0.515600
1500,0.031200,0.045316,0.496300
2000,0.022600,0.044641,0.485300
2500,0.016300,0.047926,0.453800
3000,0.008100,0.054333,0.410900



📊 [Éval #1] Exemples décodés :
  REF  : Je prefere le couscous a la place de la tarte au citron!
  PRED : Je prefere le couscous a la place de la tarte au citron.
  REF  : Puis-je avoir plus de piment dans mon la pasta ?
  PRED : Puis-je avoir plus de piment dans mon la pasta ?
  REF  : euh, je voudrais changer ma commande : au lieu de le gratin dauphinois je prends
  PRED : euh, je voudrais changer ma commande, au lieu de le gratin dauphinois je prends 
  → WER #1 = 0.4712  (400 exemples)


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}



📊 [Éval #2] Exemples décodés :
  REF  : Je prefere le couscous a la place de la tarte au citron!
  PRED : I prefer the couscous instead of the tart to the lemon.
  REF  : Puis-je avoir plus de piment dans mon la pasta ?
  PRED : Puis-je avoir plus de piment dans mon la pasta ?
  REF  : euh, je voudrais changer ma commande : au lieu de le gratin dauphinois je prends
  PRED : euh, je voudrais changer ma commande : au lieu de le gratin dauphinois je prends
  → WER #2 = 0.5156  (400 exemples)


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}



📊 [Éval #3] Exemples décodés :
  REF  : Je prefere le couscous a la place de la tarte au citron!
  PRED : Je prefere le couscous a la place de la tarte au citron.
  REF  : Puis-je avoir plus de piment dans mon la pasta ?
  PRED : Puis-je avoir plus de piment dans mon la pasta ?
  REF  : euh, je voudrais changer ma commande : au lieu de le gratin dauphinois je prends
  PRED : euh, je voudrais changer ma commande : au lieu de le gratin dauphinois je prends
  → WER #3 = 0.4963  (400 exemples)


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfi

KeyboardInterrupt: 

## Cellule 8 — Évaluation finale & WER par langue

In [ ]:
print("─" * 55)
print(f"Évaluation finale sur {len(dataset['validation']):,} exemples...")
final_results = trainer.evaluate(
    eval_dataset=dataset["validation"],
    metric_key_prefix="final"
)
print(f"✅ WER  final : {final_results.get('final_wer', 'N/A'):.4f}")
print(f"   Loss final : {final_results.get('final_loss', 'N/A'):.4f}")

# WER par langue sur test set
print(f"\nPrédictions test ({len(dataset['test']):,} exemples)...")
test_preds    = trainer.predict(dataset["test"])
pred_ids      = test_preds.predictions
label_ids     = test_preds.label_ids
label_ids_dec = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)
pred_str      = [p.strip() for p in tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)]
label_str     = [l.strip() for l in tokenizer.batch_decode(label_ids_dec, skip_special_tokens=True)]

label_str_raw = [
    tokenizer.decode([t for t in row if t != -100], skip_special_tokens=False)
    for row in label_ids
]

def detect_lang(raw_str):
    for code in ["ar", "en", "fr"]:
        if f"<|{code}|>" in raw_str:
            return code
    return "unknown"

buckets     = {"ar": ([], []), "en": ([], []), "fr": ([], []), "unknown": ([], [])}
lang_counts = {"ar": 0, "en": 0, "fr": 0, "unknown": 0}
for i, raw in enumerate(label_str_raw):
    lang = detect_lang(raw)
    lang_counts[lang] += 1
    buckets[lang][0].append(pred_str[i])
    buckets[lang][1].append(label_str[i])

print(f"\nDistribution langues : {lang_counts}")
if lang_counts["unknown"] > len(label_str_raw) * 0.1:
    print("⚠️  >10% 'unknown' — vérifie les labels du dataset")

print(f"\n{'Langue':<10} {'Exemples':>10} {'WER':>10}")
print("─" * 33)
wer_per_lang = {}
for lang in ["ar", "en", "fr"]:
    p, r  = buckets[lang]
    valid = [(pp, rr) for pp, rr in zip(p, r) if rr.strip()]
    if not valid:
        print(f"{lang:<10} {'0':>10} {'N/A':>10}")
        continue
    pv, rv = zip(*valid)
    w = wer_metric.compute(predictions=list(pv), references=list(rv))
    wer_per_lang[lang] = round(w, 4)
    print(f"{lang:<10} {len(rv):>10,} {w:>10.4f}")

if wer_per_lang:
    print(f"\n   WER moyen      : {np.mean(list(wer_per_lang.values())):.4f}")
    print(f"   Meilleure lang : {min(wer_per_lang, key=wer_per_lang.get)}")
    print(f"   Plus difficile : {max(wer_per_lang, key=wer_per_lang.get)}")

## Cellule 9 — Sauvegarde & Backup inter-sessions

In [ ]:
FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
os.makedirs(FINAL_DIR, exist_ok=True)

trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)

summary = {
    "model"        : MODEL_NAME,
    "n_gpus"       : n_gpus,
    "max_steps"    : training_args.max_steps,
    "lr"           : training_args.learning_rate,
    "batch_eff"    : effective,
    "wer_final"    : final_results.get("final_wer"),
    "loss_final"   : final_results.get("final_loss"),
    "wer_per_lang" : wer_per_lang,
}
with open(os.path.join(FINAL_DIR, "eval_results.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"✅ Modèle sauvegardé → {FINAL_DIR}/")
for fname in sorted(os.listdir(FINAL_DIR)):
    size_mb = os.path.getsize(os.path.join(FINAL_DIR, fname)) / 1e6
    print(f"   {fname:<42s} {size_mb:6.1f} Mo")

# Backup checkpoints
BACKUP_DIR = "/kaggle/working/checkpoint_backup"
shutil.copytree(OUTPUT_DIR, BACKUP_DIR, dirs_exist_ok=True)
print(f"\n📦 Backup → {BACKUP_DIR}")
print("   Settings > Output > Save as Dataset pour persister")

print("\n" + "=" * 60)
print(f"  WER final      : {final_results.get('final_wer', 'N/A')}")
print(f"  WER par langue : {wer_per_lang}")
print("=" * 60)